In [0]:
# List all secret scopes registered in the workspace
scopes = dbutils.secrets.listScopes()

print("Registered Secret Scopes:")
for scope in scopes:
    print(f" • {scope.name}")

In [0]:
# Databricks notebook source
# 1. Widgets / Base Config
dbutils.widgets.text("container", "joshuandegwa7447")
dbutils.widgets.text("storage_account", "dlspl21databricks")

_container = dbutils.widgets.get("container")
_storage_account = dbutils.widgets.get("storage_account")
_secret_scope = "default2"

# 2. Retrieve Service Principal credentials using 'default2' keys
client_id     = dbutils.secrets.get(scope=_secret_scope, key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope=_secret_scope, key="sp-databricks-adls-appkey")
tenant_id     = dbutils.secrets.get(scope=_secret_scope, key="tenant-id")

# Target Mount Path and Source URI
mount_point = "/mnt/legacy_banking"
source_uri = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net/"

# 3. OAuth 2.0 Azure SPN Configuration
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# 4. Legacy Mount Call
dbutils.fs.mount(
    source=source_uri,
    mount_point=mount_point,
    extra_configs=configs
)